# EN/DE 64k Batch Cosine Similarity

Minimal check: load `data/train_lang.csv`, sample English/German batch pairs, embed the text with the unfine-tuned base model, and compute cosine similarity between the two batch mean CLS embeddings.

Two sampling modes are compared:
- `random`: random 64k examples per language
- `balanced`: 64k examples per language, balanced across labels

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

In [ ]:
ROOT = Path("..") if Path("../data/train_lang.csv").exists() else Path(".")
TRAIN_CSV = ROOT / "data" / "train_lang.csv"

BASE_MODEL = "xlm-roberta-large"
BATCH_SIZE = 64_000
NUM_BATCHES = 10
EMBED_BATCH_SIZE = 128
MAX_LENGTH = 128
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df = pd.read_csv(TRAIN_CSV)

print(df.shape)
print(device)
display(pd.crosstab(df["lang"], df["label"]))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModel.from_pretrained(BASE_MODEL).to(device)
model.eval()

In [ ]:
rng = np.random.default_rng(SEED)
labels = sorted(df["label"].unique())
lang_indices = {lang: df.index[df["lang"].eq(lang)].to_numpy() for lang in ["eng_Latn", "deu_Latn"]}
lang_label_indices = {
    (lang, label): df.index[df["lang"].eq(lang) & df["label"].eq(label)].to_numpy()
    for lang in lang_indices
    for label in labels
}


def sample_random(lang, n=BATCH_SIZE):
    pool = lang_indices[lang]
    return rng.choice(pool, size=n, replace=len(pool) < n)


def sample_balanced(lang, n=BATCH_SIZE):
    base = n // len(labels)
    counts = np.full(len(labels), base)
    counts[: n - base * len(labels)] += 1

    parts = []
    for label, count in zip(labels, counts):
        pool = lang_label_indices[(lang, label)]
        parts.append(rng.choice(pool, size=count, replace=len(pool) < count))

    idx = np.concatenate(parts)
    rng.shuffle(idx)
    return idx


@torch.inference_mode()
def mean_embedding_for_indices(indices, desc="embed"):
    total = None
    n_seen = 0

    for start in tqdm(range(0, len(indices), EMBED_BATCH_SIZE), desc=desc, leave=False):
        batch_idx = indices[start : start + EMBED_BATCH_SIZE]
        texts = df.loc[batch_idx, "sentence"].fillna("").astype(str).tolist()
        inputs = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            cls = model(**inputs).last_hidden_state[:, 0, :]

        batch_sum = cls.float().sum(dim=0).cpu()
        total = batch_sum if total is None else total + batch_sum
        n_seen += len(batch_idx)

    return total / n_seen


def mean_cosine(en_idx, de_idx, desc):
    en_mean = mean_embedding_for_indices(en_idx, desc=f"{desc} eng")
    de_mean = mean_embedding_for_indices(de_idx, desc=f"{desc} deu")
    return F.cosine_similarity(en_mean, de_mean, dim=0).item()


def run_batches(mode, num_batches=NUM_BATCHES):
    sampler = sample_balanced if mode == "balanced" else sample_random
    rows = []
    for batch in range(1, num_batches + 1):
        en_idx = sampler("eng_Latn")
        de_idx = sampler("deu_Latn")
        rows.append(
            {
                "mode": mode,
                "batch": batch,
                "n_eng": len(en_idx),
                "n_deu": len(de_idx),
                "cosine": mean_cosine(en_idx, de_idx, desc=f"{mode} {batch}"),
            }
        )
        display(pd.DataFrame(rows).tail(1))
    return pd.DataFrame(rows)

In [ ]:
results = pd.concat(
    [run_batches("random"), run_batches("balanced")],
    ignore_index=True,
)

display(results)
display(results.groupby("mode")["cosine"].agg(["mean", "std", "min", "max"]))